# Voice Pipeline Run

Train the voice emotion model and run a sample prediction.

Steps:
- Train the voice emotion model.
- Run a sample prediction on available audio.
- Verify model artifacts.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'model': {},
    'prediction': None,
}

run([PY, 'app/models/voice/emotion_train.py', '--limit-per-class', '0'])


In [ ]:
# Run a sample prediction if audio exists.
from app.models.voice.emotion_predict import predict_emotion

voice_root = REPO_ROOT / 'data' / 'raw' / 'voice'
if voice_root.exists():
    wav_files = [p for p in voice_root.rglob('*.wav')]
    if wav_files:
        sample = wav_files[0]
        audio_bytes = sample.read_bytes()
        result = predict_emotion(audio_bytes=audio_bytes, filename=sample.name)
        summary['prediction'] = result
        print(result)
    else:
        print('No wav files found under', voice_root)
else:
    print('Missing:', voice_root)


In [ ]:
# Verify model artifacts.
model_paths = [
    REPO_ROOT / 'models' / 'voice_emotion.pkl',
    REPO_ROOT / 'models' / 'voice_emotion_nn.pt',
]
for path in model_paths:
    if path.exists():
        summary['model'] = summary.get('model', {})
        summary['model'][path.name] = round(path.stat().st_size / 1024**2, 2)
        print(path.name, summary['model'][path.name], 'MB')
    else:
        print('Missing:', path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'execution_voice_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
